<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 02 · AI INTEGRATIONS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Run Doris AI Functions with a Local Qwen Model</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Connect the existing single-FE/single-BE Doris container to local Ollama, validate model output, materialize reusable results, and return to ordinary SQL.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">12 steps · Local model · About 30 minutes</span>
</div>

**Module 2: Use Doris AI Functions Safely** · [Module outline](readme.md)

You will verify both network paths to the model, load a bounded bilingual dataset, create one local AI Resource, exercise the core AI Functions, validate the non-deterministic output, and persist selected values for normal SQL analysis.

> **Run order:** execute every cell from top to bottom. The Lab owns the resource `ollama_qwen25_1_5b` and the two `doris_course.ai_*` tables. Do not run two copies of this notebook against the same Doris instance at the same time.

## Before you start

This notebook assumes that the course prerequisites have already been completed:

- Docker Desktop is running.
- The course container named `doris` provides **one FE and one BE** using Apache Doris 4.1.3.
- Ollama is running on macOS and already contains `qwen2.5:1.5b-instruct`.
- The Doris container can resolve `host.docker.internal` and reach the configured Ollama port on the host (default: `11434`).

The notebook performs read-only preflight checks before it creates any Doris object. If a check
fails, stop at that cell and fix the environment instead of skipping the assertion.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
MODULE_DIR = COURSE_ROOT / "level1" / "module02-doris-ai-functions"

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import AiFunctionsLab
from doris_course.ai_functions import MODEL_NAME, RESOURCE_NAME
from doris_labkit.results import show_frame
from doris_labkit.ui import card, show_sql

OLLAMA_PORT = 11434  # Change this if Ollama listens on another host port.

lab = AiFunctionsLab(lab_dir=MODULE_DIR, ollama_port=OLLAMA_PORT)

## 1. Verify the Doris environment

The AI request is sent by a Doris BE, not by this notebook. We therefore verify both the SQL
service and the expected one-FE/one-BE topology before testing Ollama.

In [ ]:
lab.start()
lab.connect()
lab.execute("USE doris_course")
lab.execute("SET query_timeout = 180")

version_text = lab.query("SELECT VERSION() AS mysql_compatible_version").iloc[0, 0]
frontends = lab.query("SHOW FRONTENDS")
backends = lab.query("SHOW BACKENDS")

alive_frontends = frontends[frontends["Alive"].astype(str).str.lower() == "true"]
alive_backends = backends[backends["Alive"].astype(str).str.lower() == "true"]

assert len(alive_frontends) == 1, f"Expected one alive FE, found {len(alive_frontends)}"
assert len(alive_backends) == 1, f"Expected one alive BE, found {len(alive_backends)}"
assert alive_frontends["Version"].astype(str).str.contains("4.1.3", regex=False).all()
assert alive_backends["Version"].astype(str).str.contains("4.1.3", regex=False).all()

environment = pd.DataFrame(
    [
        {"check": "SQL endpoint", "result": "connected", "expected": "connected"},
        {"check": "Alive FE count", "result": len(alive_frontends), "expected": 1},
        {"check": "Alive BE count", "result": len(alive_backends), "expected": 1},
        {"check": "SELECT VERSION()", "result": version_text, "expected": "MySQL-compatible value"},
    ]
)
show_frame("Doris environment preflight", environment)

**Expected result**

A green container status and a four-row preflight table. Exactly one FE and one BE are alive, and both report Doris 4.1.3.

## 2. Verify Ollama from both network paths

There are two distinct paths:

- **Notebook → host Ollama:** `http://127.0.0.1:<OLLAMA_PORT>`
- **Doris BE container → host Ollama:** `http://host.docker.internal:<OLLAMA_PORT>`

The next two cells use direct OpenAI-compatible HTTP requests only as preflight checks. They do
not call a Doris AI Function and therefore do not prove the Doris Resource is configured yet.

In [ ]:
host_check, host_reply, model_digest = lab.ollama_host_preflight()
show_frame("Host-to-Ollama preflight", host_check)
print("Direct model reply:", host_reply)
print("Pinned model digest:", model_digest)

**Expected result**

The host route succeeds, the reply contains `HOST_OK`, and the installed model digest is printed.

In [ ]:
be_check, be_reply = lab.ollama_be_preflight()
show_frame("BE-container-to-Ollama preflight", be_check)
print("Direct model reply from the BE network path:", be_reply)

**Expected result**

The container route through `host.docker.internal` succeeds and the reply contains `BE_OK`. This is the network path the Doris BE will use.

## 3. Load the support-ticket dataset

The CSV contains deliberately small, synthetic examples in English and Chinese. Ground-truth
columns are included for teaching and evaluation; they are not passed into the model.

The notebook recreates only its own two tables. The raw text is retained so that every derived
AI value remains auditable.

In [ ]:
lab.execute("DROP TABLE IF EXISTS doris_course.ai_ticket_enrichment")
lab.execute("DROP TABLE IF EXISTS doris_course.ai_support_tickets")

lab.execute("""
CREATE TABLE doris_course.ai_support_tickets (
    ticket_id BIGINT NOT NULL,
    created_at DATETIME NOT NULL,
    language VARCHAR(8) NOT NULL,
    raw_text STRING NOT NULL,
    expected_topic VARCHAR(32) NOT NULL,
    expected_sentiment VARCHAR(16) NOT NULL
)
DUPLICATE KEY(ticket_id)
DISTRIBUTED BY HASH(ticket_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("""
CREATE TABLE doris_course.ai_ticket_enrichment (
    ticket_id BIGINT NOT NULL,
    topic VARCHAR(32),
    sentiment VARCHAR(16),
    summary STRING,
    model_name VARCHAR(128) NOT NULL,
    model_digest VARCHAR(128) NOT NULL,
    prompt_version VARCHAR(32) NOT NULL,
    processed_at DATETIME NOT NULL,
    run_id VARCHAR(64) NOT NULL
)
UNIQUE KEY(ticket_id)
DISTRIBUTED BY HASH(ticket_id) BUCKETS 1
PROPERTIES (
    "replication_num" = "1",
    "enable_unique_key_merge_on_write" = "true"
)
""")

ticket_csv = MODULE_DIR / "data" / "ai_support_tickets.csv"
tickets = pd.DataFrame(lab.read_ticket_csv(ticket_csv))
loaded_rows = lab.load_tickets(ticket_csv)
assert loaded_rows == len(tickets)

show_frame(f"Loaded {loaded_rows} synthetic support tickets", tickets.head(8))

**Expected result**

The preview shows eight of the validated synthetic tickets and reports that all rows from the CSV were loaded.

In [ ]:
dataset_profile = lab.query("""
SELECT
    language,
    expected_topic,
    COUNT(*) AS ticket_count,
    ROUND(AVG(CHAR_LENGTH(raw_text)), 1) AS avg_characters
FROM doris_course.ai_support_tickets
GROUP BY language, expected_topic
ORDER BY language, expected_topic
""")

row_count = int(lab.query(
    "SELECT COUNT(*) AS row_count FROM doris_course.ai_support_tickets"
).iloc[0, 0])
assert row_count == len(tickets)
show_frame("Input distribution", dataset_profile)

**Expected result**

The grouped profile covers English and Chinese tickets. The row-count assertion passes without output.

### Bound the model workload before calling AI

The main path enriches only six tickets. The anti-join makes the candidate set idempotent: after
successful materialization, rerunning the candidate query returns zero rows instead of spending
tokens again.

In [ ]:
CANDIDATE_IDS = (1001, 1002, 1003, 1004, 1005, 1009)

candidate_sql = f"""
SELECT s.ticket_id, s.language, s.raw_text, s.expected_topic, s.expected_sentiment
FROM doris_course.ai_support_tickets AS s
LEFT ANTI JOIN doris_course.ai_ticket_enrichment AS e
    ON s.ticket_id = e.ticket_id
WHERE s.ticket_id IN ({", ".join(map(str, CANDIDATE_IDS))})
ORDER BY s.ticket_id
"""
candidates = lab.query(candidate_sql)
assert len(candidates) == 6, f"Expected six new candidates, found {len(candidates)}"
show_sql("Candidate selection SQL", candidate_sql)
show_frame("Bounded enrichment candidates", candidates)

call_budget = pd.DataFrame(
    [
        {"step": "AI_GENERATE smoke test", "logical_model_calls": 1},
        {"step": "AI_CLASSIFY on six rows", "logical_model_calls": 6},
        {"step": "AI_SENTIMENT on six rows", "logical_model_calls": 6},
        {"step": "AI_EXTRACT on two rows", "logical_model_calls": 2},
        {"step": "AI_SUMMARIZE on two rows", "logical_model_calls": 2},
        {"step": "AI_AGG on two groups", "logical_model_calls": 2},
    ]
)
show_frame("Default main-path budget: 19 logical model calls", call_budget)

**Expected result**

Six candidates are shown, followed by a budget of 19 logical model calls for the default path.

## 4. Create a local Doris AI Resource

The Resource stores provider configuration, not model weights. `ai.endpoint` must use the address
visible **from the BE container**, so it differs from the host-side preflight URL.

This lab uses no secret. In a production API-key configuration, use a secret-management workflow
and never print the key with `SHOW RESOURCES` or commit it to a notebook.

In [ ]:
existing_resource = lab.assert_owned_resource_or_absent()
if not existing_resource.empty:
    lab.execute(f'DROP RESOURCE "{RESOURCE_NAME}"')

resource_sql = f"""
CREATE RESOURCE "{RESOURCE_NAME}"
PROPERTIES (
    "type" = "ai",
    "ai.provider_type" = "local",
    "ai.endpoint" = "{lab.be_ollama}{lab.chat_path}",
    "ai.model_name" = "{MODEL_NAME}",
    "ai.temperature" = "0",
    "ai.max_token" = "256",
    "ai.max_retries" = "1",
    "ai.retry_delay_second" = "1"
)
"""
show_sql("Create the local AI Resource", resource_sql)
lab.execute(resource_sql)
lab.execute(f'SET default_ai_resource = "{RESOURCE_NAME}"')

created_resource = lab.assert_owned_resource_or_absent()
safe_resource_view = created_resource[
    created_resource["Item"].isin(
        ["type", "ai.provider_type", "ai.endpoint", "ai.model_name", "ai.temperature", "ai.max_token"]
    )
]
show_frame("Validated local AI Resource", safe_resource_view)

**Expected result**

The Resource reports provider `local`, model `qwen2.5:1.5b-instruct`, and an endpoint using `host.docker.internal` with the configured `OLLAMA_PORT`.

## 5. Smoke-test the end-to-end Doris path with `AI_GENERATE`

Unlike the two direct HTTP checks, this query travels through Doris and the Resource. The
assertion checks a semantic marker rather than exact equality.

In [ ]:
generate_result = lab.query(f"""
SELECT AI_GENERATE(
    '{RESOURCE_NAME}',
    'Reply with the marker LOCAL_OK and no other words.'
) AS model_reply
""")

generate_reply = str(generate_result.iloc[0]["model_reply"]).strip()
assert generate_reply, "AI_GENERATE returned an empty value"
assert "LOCAL_OK" in generate_reply.upper(), f"Expected LOCAL_OK marker, got: {generate_reply!r}"
show_frame("AI_GENERATE smoke-test result", generate_result)

**Expected result**

One non-empty result containing the marker `LOCAL_OK`. The exact surrounding wording may vary.

## 6. Produce finite labels with `AI_CLASSIFY`

Finite labels are easier to validate than free text. The label set is deliberately small and
domain-specific. We validate membership first; agreement with the synthetic ground truth is an
evaluation signal, not a reason to crash the lab.

In [ ]:
def classify_one(ticket_id):
    return f"""
    SELECT
        ticket_id,
        raw_text,
        expected_topic,
        AI_CLASSIFY(
            '{RESOURCE_NAME}',
            raw_text,
            ['refund', 'shipping', 'quality', 'account', 'product_question']
        ) AS actual_topic
    FROM doris_course.ai_support_tickets
    WHERE ticket_id = {int(ticket_id)}
    """

show_sql("AI_CLASSIFY pattern (repeated for six bounded IDs)", classify_one(CANDIDATE_IDS[0]))
classification = pd.concat(
    [lab.query(classify_one(ticket_id)) for ticket_id in CANDIDATE_IDS],
    ignore_index=True,
).sort_values("ticket_id")

valid_topics = {"refund", "shipping", "quality", "account", "product_question"}
classification["valid_label"] = classification["actual_topic"].isin(valid_topics)
classification["matches_expected"] = (
    classification["actual_topic"] == classification["expected_topic"]
)

assert len(classification) == 6
assert classification["actual_topic"].notna().all()
assert classification["valid_label"].all(), "AI_CLASSIFY returned a label outside the allowed set"
show_frame("Validated topic labels", classification)

**Expected result**

Six rows are shown and every value in `valid_label` is `True`. A mismatch with the teaching label is allowed and remains visible.

## 7. Produce sentiment labels with `AI_SENTIMENT`

The function returns a finite label, but sentiment is still domain-dependent. Sarcasm, mixed
messages, and multilingual input can produce reasonable disagreement. Scalar calls are issued one
ticket at a time because the 1.5B teaching model is less reliable at returning correctly sized
multi-item batches; this trades throughput for predictable parsing.

In [ ]:
def sentiment_one(ticket_id):
    return f"""
    SELECT
        ticket_id,
        raw_text,
        expected_sentiment,
        AI_SENTIMENT('{RESOURCE_NAME}', raw_text) AS actual_sentiment
    FROM doris_course.ai_support_tickets
    WHERE ticket_id = {int(ticket_id)}
    """

sentiment = pd.concat(
    [lab.query(sentiment_one(ticket_id)) for ticket_id in CANDIDATE_IDS],
    ignore_index=True,
).sort_values("ticket_id")

valid_sentiments = {"positive", "negative", "neutral", "mixed"}
sentiment["normalized_sentiment"] = sentiment["actual_sentiment"].str.lower().str.strip()
sentiment["valid_label"] = sentiment["normalized_sentiment"].isin(valid_sentiments)
sentiment["matches_expected"] = (
    sentiment["normalized_sentiment"] == sentiment["expected_sentiment"]
)

assert len(sentiment) == 6
assert sentiment["actual_sentiment"].notna().all()
assert sentiment["valid_label"].all(), "AI_SENTIMENT returned an unsupported label"
show_frame("Validated sentiment labels", sentiment)

**Expected result**

Six normalized sentiment labels are shown and every value in `valid_label` is `True`. Domain-dependent disagreements remain visible.

## 8. Extract facts and summarize text

`AI_EXTRACT` returns model-generated text describing requested fields. Treat that output as a
candidate fact set: parse and validate it before it drives automation. Missing source facts should
remain missing rather than being invented.

`AI_SUMMARIZE` creates useful display text, but the raw ticket remains the source of truth.

In [ ]:
def extract_one(ticket_id):
    return f"""
    SELECT
        ticket_id,
        raw_text,
        AI_EXTRACT(
            '{RESOURCE_NAME}',
            raw_text,
            ['order_id', 'product', 'request']
        ) AS extracted_fields
    FROM doris_course.ai_support_tickets
    WHERE ticket_id = {int(ticket_id)}
    """

extract_result = pd.concat(
    [lab.query(extract_one(ticket_id)) for ticket_id in (1002, 1011)],
    ignore_index=True,
).sort_values("ticket_id")

assert len(extract_result) == 2
assert extract_result["extracted_fields"].notna().all()
assert extract_result["extracted_fields"].str.strip().ne("").all()
show_frame("AI_EXTRACT candidate facts", extract_result)

**Expected result**

Two non-empty candidate fact sets are displayed beside their raw ticket text. Treat these values as unverified model output.

In [ ]:
def summarize_one(ticket_id):
    return f"""
    SELECT
        ticket_id,
        raw_text,
        AI_SUMMARIZE('{RESOURCE_NAME}', raw_text) AS summary
    FROM doris_course.ai_support_tickets
    WHERE ticket_id = {int(ticket_id)}
    """

summary_result = pd.concat(
    [lab.query(summarize_one(ticket_id)) for ticket_id in (1002, 1010)],
    ignore_index=True,
).sort_values("ticket_id")

assert len(summary_result) == 2
assert summary_result["summary"].notna().all()
assert summary_result["summary"].str.strip().ne("").all()
show_frame("AI_SUMMARIZE output beside raw text", summary_result)

**Expected result**

Two non-empty summaries are displayed beside the raw text. Exact wording is intentionally not asserted.

## 9. Optional function gallery

The default lab path keeps this gallery disabled so that setup validation does not unexpectedly
multiply model calls. Set `RUN_OPTIONAL_GALLERY = True` only when you want to inspect these
additional functions.

- `AI_FILTER` and `AI_SIMILARITY` are model judgments, not access-control rules or exact numeric
  thresholds.
- `AI_TRANSLATE`, `AI_FIXGRAMMAR`, and `AI_MASK` transform text. Retain the raw value; masking by
  a language model is not a compliance-grade DLP control.

In [ ]:
RUN_OPTIONAL_GALLERY = False

if not RUN_OPTIONAL_GALLERY:
    card(
        "Set RUN_OPTIONAL_GALLERY = True and rerun this cell to spend five additional logical model calls.",
        "info",
        "Optional gallery skipped",
    )
else:
    optional_queries = {
        "AI_FILTER": f"""
            SELECT AI_FILTER(
                '{RESOURCE_NAME}',
                CONCAT('The customer explicitly asks for a refund: ', raw_text)
            ) AS result
            FROM doris_course.ai_support_tickets WHERE ticket_id = 1001
        """,
        "AI_SIMILARITY": f"""
            SELECT AI_SIMILARITY(
                '{RESOURCE_NAME}',
                'Where is my shipment?',
                raw_text
            ) AS result
            FROM doris_course.ai_support_tickets WHERE ticket_id = 1002
        """,
        "AI_TRANSLATE": f"""
            SELECT AI_TRANSLATE(
                '{RESOURCE_NAME}',
                raw_text,
                'English'
            ) AS result
            FROM doris_course.ai_support_tickets WHERE ticket_id = 1005
        """,
        "AI_FIXGRAMMAR": f"""
            SELECT AI_FIXGRAMMAR(
                '{RESOURCE_NAME}',
                'my package have not arrive yet'
            ) AS result
        """,
        "AI_MASK": f"""
            SELECT AI_MASK(
                '{RESOURCE_NAME}',
                'Customer Alice uses alice@example.com and order A-10011.',
                ['name', 'email', 'order_id']
            ) AS result
        """,
    }
    optional_results = []
    for function_name, sql in optional_queries.items():
        value = lab.query(sql).iloc[0, 0]
        optional_results.append({"function": function_name, "result": value})
    show_frame("Optional AI Function gallery", pd.DataFrame(optional_results))

**Expected result**

By default, an information card says the optional gallery was skipped. If enabled, one result is shown for each of the five optional functions.

## 10. Materialize validated results

Repeated AI calls are expensive and non-deterministic. We now write the already validated Python
results into a Doris table—without calling the model again—together with model and run metadata.

The summary is materialized only for ticket `1002`; the other candidates preserve `NULL`, making
the difference between “not generated” and an empty string explicit.

In [ ]:
classification_for_write = classification[["ticket_id", "actual_topic"]].copy()
sentiment_for_write = sentiment[["ticket_id", "normalized_sentiment"]].copy()
summary_for_write = summary_result[
    summary_result["ticket_id"].isin(CANDIDATE_IDS)
][["ticket_id", "summary"]].copy()

enrichment = (
    candidates[["ticket_id"]]
    .merge(classification_for_write, on="ticket_id", how="left", validate="one_to_one")
    .merge(sentiment_for_write, on="ticket_id", how="left", validate="one_to_one")
    .merge(summary_for_write, on="ticket_id", how="left", validate="one_to_one")
    .rename(
        columns={
            "actual_topic": "topic",
            "normalized_sentiment": "sentiment",
        }
    )
)

assert enrichment["topic"].notna().all()
assert enrichment["sentiment"].notna().all()

show_sql("Materialization INSERT template", """
INSERT INTO doris_course.ai_ticket_enrichment
    (ticket_id, topic, sentiment, summary, model_name, model_digest,
     prompt_version, processed_at, run_id)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

run_id = lab.write_enrichment(
    enrichment.to_dict("records"),
    model_digest=model_digest,
    prompt_version="lab-v1",
)
print("Materialization run_id:", run_id)

**Expected result**

A new `run_id` is printed after six already-validated rows are written. This step does not make another model call.

In [ ]:
materialized = lab.query("""
SELECT
    ticket_id,
    topic,
    sentiment,
    summary,
    model_name,
    LEFT(model_digest, 20) AS model_digest_prefix,
    prompt_version,
    run_id
FROM doris_course.ai_ticket_enrichment
ORDER BY ticket_id
""")
assert len(materialized) == 6

remaining_candidates = lab.query(candidate_sql)
assert remaining_candidates.empty, "The anti-join should return zero candidates after materialization"
show_frame("Materialized and traceable AI enrichment", materialized)
card(
    "The candidate anti-join now returns zero rows, so a rerun can avoid duplicate model calls.",
    "ok",
    "Idempotency check passed",
)

**Expected result**

Six traceable enrichment rows are shown and a green card confirms that the anti-join now returns zero candidates.

In [ ]:
analytics = lab.query("""
SELECT
    topic,
    sentiment,
    COUNT(*) AS ticket_count
FROM doris_course.ai_ticket_enrichment
GROUP BY topic, sentiment
ORDER BY ticket_count DESC, topic, sentiment
""")
show_frame("Ordinary SQL over materialized AI labels", analytics)

**Expected result**

A normal SQL aggregation counts the materialized rows by topic and sentiment; no AI Function is called.

## 11. Summarize bounded groups with `AI_AGG`

`AI_AGG` sends multiple rows from each SQL group to the model. Group size is therefore a token
budget decision, not merely a SQL decision. This example deliberately selects at most two rows for
each of two topics, then checks only that every group returns non-empty text.

In [ ]:
aggregate_input = lab.query("""
WITH ranked AS (
    SELECT
        ticket_id,
        expected_topic,
        raw_text,
        ROW_NUMBER() OVER (PARTITION BY expected_topic ORDER BY ticket_id) AS row_num
    FROM doris_course.ai_support_tickets
    WHERE expected_topic IN ('shipping', 'quality')
)
SELECT ticket_id, expected_topic, raw_text
FROM ranked
WHERE row_num <= 2
ORDER BY expected_topic, ticket_id
""")
show_frame("Bounded rows passed to AI_AGG", aggregate_input)

**Expected result**

At most two source rows are displayed for each of the `quality` and `shipping` groups.

In [ ]:
def aggregate_one(topic):
    assert topic in {"shipping", "quality"}
    return f"""
    WITH ranked AS (
        SELECT
            ticket_id,
            expected_topic,
            raw_text,
            ROW_NUMBER() OVER (PARTITION BY expected_topic ORDER BY ticket_id) AS row_num
        FROM doris_course.ai_support_tickets
        WHERE expected_topic = '{topic}'
    )
    SELECT
        expected_topic,
        AI_AGG(
            '{RESOURCE_NAME}',
            raw_text,
            'Summarize the shared customer issue in one short sentence. Do not invent facts.'
        ) AS group_summary
    FROM ranked
    WHERE row_num <= 2
    GROUP BY expected_topic
    """

aggregate_result = pd.concat(
    [lab.query(aggregate_one(topic)) for topic in ("quality", "shipping")],
    ignore_index=True,
).sort_values("expected_topic")

assert len(aggregate_result) == 2
assert aggregate_result["group_summary"].notna().all()
assert aggregate_result["group_summary"].str.strip().ne("").all()
show_frame("AI_AGG summaries for two bounded groups", aggregate_result)

**Expected result**

One non-empty summary is returned for `quality` and one for `shipping`. Inspect each result for omitted minority cases.

## 12. Remove the AI Resource

The lab removes the connection object but intentionally retains both Doris tables and the Ollama
model. This lets later exercises inspect the raw and materialized data without re-running AI calls.

In [ ]:
lab.assert_owned_resource_or_absent()
lab.execute(f'DROP RESOURCE IF EXISTS "{RESOURCE_NAME}"')
assert lab.resource_properties().empty, "The lab-owned AI Resource still exists"

retained_counts = lab.query("""
SELECT 'ai_support_tickets' AS table_name, COUNT(*) AS row_count
FROM doris_course.ai_support_tickets
UNION ALL
SELECT 'ai_ticket_enrichment' AS table_name, COUNT(*) AS row_count
FROM doris_course.ai_ticket_enrichment
ORDER BY table_name
""")
show_frame("Cleanup complete: tables retained, AI Resource removed", retained_counts)

**Expected result**

The AI Resource is absent. Both teaching tables remain and their retained row counts are displayed.

<div style="background: #ecfdf5; border-left: 5px solid #0b7a53; padding: 20px 24px; border-radius: 8px; margin-top: 18px;">
  <h2 style="margin-top: 0; color: #075b42;">Lab complete</h2>
  <p style="margin-bottom: 8px;">You have completed a bounded, observable AI enrichment workflow:</p>
  <ul style="margin-bottom: 0;">
    <li>verified both host and BE-container network paths to the local model;</li>
    <li>created and smoke-tested a Doris AI Resource;</li>
    <li>validated finite labels and non-empty generated text without asserting exact wording;</li>
    <li>materialized traceable results and reused them with ordinary SQL;</li>
    <li>bounded cross-row aggregation and removed the temporary Resource.</li>
  </ul>
</div>

The key engineering boundary is simple: an AI Function can produce a useful candidate value, but
SQL types, allowed-label checks, ground truth, provenance, and application policy still decide
whether that value is safe to use.

### Reference documentation

- [Apache Doris AI Functions](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/ai-functions/)
- [Apache Doris AI Resource](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/resource-management/CREATE-RESOURCE/)
- [Ollama OpenAI compatibility](https://docs.ollama.com/openai)